In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV, LassoCV
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

In [11]:
X_train = pd.read_csv("../../data/processed/X_train.csv")
X_test = pd.read_csv("../../data/processed/X_test.csv")
y_train = pd.read_csv("../../data/processed/y_train.csv", header=None).squeeze()
y_test = pd.read_csv("../../data/processed/y_test.csv", header=None).squeeze()

In [12]:
ridge_alphas = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0]
lasso_alphas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
xgb_params = {
    'xgb__n_estimators': [100, 200, 300, 500],
    'xgb__max_depth': [3, 5, 7, 9],
    'xgb__learning_rate': [0.01, 0.05, 0.1, 0.3],
    'xgb__subsample': [0.7, 0.8, 0.9, 1.0],
    'xgb__colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}
rf_params = {
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [10, 20, 30],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

In [13]:
models_to_tune = {
     "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", RidgeCV(alphas=ridge_alphas, cv=5))
    ]),
     "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LassoCV(alphas=lasso_alphas, cv=5))
    ]),
     "XGBRegressor": GridSearchCV(
         Pipeline([("xgb", XGBRegressor(random_state=42))]),
            param_grid=xgb_params,
            scoring="neg_mean_squared_error",
            cv=5,
            n_jobs=-1
     ),
     "RandomForestRegressor": GridSearchCV(
         Pipeline([("rf", RandomForestRegressor(random_state=42))]),
        param_grid=rf_params,
        scoring="neg_mean_squared_error",
        cv=5,
        n_jobs=-1
     )
}

In [14]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [15]:
results = []
best_r2 = -float("inf")
best_model = None
best_name = None

In [16]:
for model_name, model in models_to_tune.items():
    model.fit(X_train, y_train)
    y_pred_model = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_model)
    mae = mean_absolute_error(y_test, y_pred_model)
    r2 = r2_score(y_test, y_pred_model)

    if r2 > best_r2:
        best_r2 = r2
        best_model = model
        best_name = model_name
        
    results.append({
        "Model": model_name,
        "MSE": mse,
        "MAE": mae,
        "R2": r2
    })

In [21]:
import os
import joblib
os.makedirs("../../models", exist_ok=True)

# Saved Best Tuned Model
joblib.dump(best_model, f"../../models/best_model.pkl")
print(f"Models saved {best_name} to models/best_model.pkl")

Models saved XGBRegressor to models/best_model.pkl


In [18]:
tuned_models_result = pd.DataFrame(results)
tuned_models_result

,Model,MSE,MAE,R2
0,Ridge,2.420591e+10,116237.054532,0.660753
1,Lasso,2.418836e+10,116157.433173,0.660999
2,XGBRegressor,2.306873e+10,112595.492188,0.676691
3,RandomForestRegressor,2.542566e+10,118123.530678,0.643658


In [22]:
os.makedirs("../../reports", exist_ok=True)

# Save tuned models result
tuned_models_result.to_csv("../../reports/02_tuned_result.csv", index=False)